# 🥈 Notebook 02 — Silver Processing
**Layer**: Silver — clean, enrich, HCC-map
**Inputs**: `bronze.raw_claims`, `bronze.raw_members`
**Outputs**:
- `silver.clean_claims` — deduplicated, 9-flag quality-validated claims
- `silver.hcc_mapped_claims` — CMS HCC v28 enriched with RAF coefficients + interaction terms
- `silver.member_profile` — demographics with demographic RAF coefficients
**Runtime**: ~10–15 min for ~1M claim lines

## Serverless architecture for HCC UDFs
`spark.sparkContext.addPyFile()` is blocked on Databricks Serverless.
The correct pattern is **`@pandas_udf` closures**:
1. Import the HCC mapping dict from `src/silver/hcc_mapper.py` **on the driver**
2. Assign it to a module-level variable (`_HCC`)
3. Define `@pandas_udf` functions that **close over** `_HCC`
4. Spark serializes the entire closure (function + captured `_HCC`) to each worker
5. No JVM access, no `sparkContext` — fully Serverless-compatible
6. Arrow-optimized batch processing — faster than row-level Python UDFs


## 1. Setup

In [0]:
import sys, os

# Auto-detect repo root — works for any user / workspace path
_nb_path  = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
repo_root = "/Workspace" + "/".join(_nb_path.split("/")[:-2])

if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

CATALOG       = "medicare_lakehouse"
BRONZE_SCHEMA = "bronze"
SILVER_SCHEMA = "silver"
GOLD_SCHEMA   = "gold"
ML_SCHEMA     = "ml_features"
VOLUME_PATH   = f"/Volumes/{CATALOG}/default/landing_zone/raw"

print(f"Repo root  : {repo_root}")
print(f"Catalog    : {CATALOG}")
print(f"Volume path: {VOLUME_PATH}")
_ok = os.path.exists(os.path.join(repo_root, "src"))
print("src/ found : ✅" if _ok else "❌ src/ missing — check repo clone")


## 2. Silver schema

In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SILVER_SCHEMA}")
print(f"✅ Schema: {CATALOG}.{SILVER_SCHEMA}")


## 3. HCC v28 UDFs — pandas_udf closures (Serverless-native)
Import HCC catalogue on driver → capture in `_HCC` → closures serialize to workers.


In [0]:
from src.silver.hcc_mapper import HCC_MAPPING, DEMOGRAPHIC_RAF
from pyspark.sql.functions import pandas_udf
from pyspark.sql.types import ArrayType, IntegerType, DoubleType, StringType, BooleanType
import pandas as pd

# Capture at module scope — closures serialize these to Spark workers
_HCC      = HCC_MAPPING
_DEMO_RAF = DEMOGRAPHIC_RAF

print(f"HCC v28 catalogue loaded: {len(_HCC)} ICD-10 codes")
print(f"Demographic RAF cells   : {len(_DEMO_RAF)} age×sex combinations")

# ── UDF 1: ICD-10 code string → sorted list of unique HCC numbers ─────────
@pandas_udf(ArrayType(IntegerType()))
def icd_to_hcc_list(codes_series: pd.Series) -> pd.Series:
    def _map(s):
        if not s or pd.isna(s): return []
        seen, result = set(), []
        for code in str(s).split("|"):
            m = _HCC.get(code.strip().upper())
            if m and m[0] not in seen:
                seen.add(m[0]); result.append(m[0])
        return sorted(result)
    return codes_series.apply(_map)

# ── UDF 2: ICD-10 string → sum of unique RAF coefficients (no double-count) ─
@pandas_udf(DoubleType())
def icd_to_raf_total(codes_series: pd.Series) -> pd.Series:
    def _total(s):
        if not s or pd.isna(s): return 0.0
        seen, total = set(), 0.0
        for code in str(s).split("|"):
            m = _HCC.get(code.strip().upper())
            if m and m[0] not in seen:
                seen.add(m[0]); total += m[2]
        return round(total, 4)
    return codes_series.apply(_total)

# ── UDF 3: Primary ICD-10 → HCC number ────────────────────────────────────
@pandas_udf(IntegerType())
def icd_to_primary_hcc(code_series: pd.Series) -> pd.Series:
    def _hcc(s):
        if not s or pd.isna(s): return None
        m = _HCC.get(str(s).strip().upper())
        return m[0] if m else None
    return code_series.apply(_hcc)

# ── UDF 4: Primary ICD-10 → human-readable description ────────────────────
@pandas_udf(StringType())
def icd_to_primary_hcc_desc(code_series: pd.Series) -> pd.Series:
    def _desc(s):
        if not s or pd.isna(s): return "Unknown"
        m = _HCC.get(str(s).strip().upper())
        return m[1] if m else "Not Mapped"
    return code_series.apply(_desc)

# ── UDF 5: Demographic RAF (age × sex lookup table) ────────────────────────
@pandas_udf(DoubleType())
def demographic_raf_udf(sex_s: pd.Series, age_s: pd.Series) -> pd.Series:
    def _raf(row):
        sex, age = row
        if pd.isna(sex) or pd.isna(age): return 0.45
        ab = min(range(65, 100, 5), key=lambda a: abs(a - max(65, min(int(age), 95))))
        return _DEMO_RAF.get((str(sex), ab), 0.45)
    return pd.Series([_raf(r) for r in zip(sex_s, age_s)])

print("\n✅ All 5 pandas_udf closures registered — Serverless-compatible")
print("   Workers receive HCC dict via closure serialization (no sparkContext needed)")


## 4. Load Bronze

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

bronze_claims  = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.raw_claims")
bronze_members = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.raw_members")

print(f"bronze.raw_claims  : {bronze_claims.count():,} rows")
print(f"bronze.raw_members : {bronze_members.count():,} rows")


## 5. Clean claims — deduplication + 9 quality flags

**Dedup logic**: `ROW_NUMBER()` over `(claim_id, bene_id, service_date)` ordered by
`DESC paid_amount` — highest paid amount wins. Replicates real-world claim re-submission
where corrected claims carry higher payment.

**Quality framework**: Failed rows are **retained** in `clean_claims` for audit review.
They are excluded from HCC mapping and Gold aggregations via `_quality_pass = true` filter.
This is intentional — you can always investigate why specific claims failed.


In [0]:
# Cast types (CSV reads everything as string)
claims_typed = (
    bronze_claims
    .withColumn("service_date",   F.to_date("service_date", "yyyy-MM-dd"))
    .withColumn("claim_amount",   F.col("claim_amount").cast("double"))
    .withColumn("allowed_amount", F.col("allowed_amount").cast("double"))
    .withColumn("paid_amount",    F.col("paid_amount").cast("double"))
    .withColumn("claim_year",     F.coalesce(F.col("claim_year"),  F.year("service_date")))
    .withColumn("claim_month",    F.coalesce(F.col("claim_month"), F.month("service_date")))
)

# Deduplication — single ROW_NUMBER window, keep max paid_amount
dedup_win = (
    Window.partitionBy("claim_id", "bene_id", "service_date")
          .orderBy(F.desc("paid_amount"))
)
claims_deduped = (
    claims_typed
    .withColumn("_row_rank", F.row_number().over(dedup_win))
    .filter(F.col("_row_rank") == 1)
    .drop("_row_rank")
)

# 9 quality flags — computed as a single transform pass (lazy DAG)
FLAG_COLS = [
    "flag_null_claim_id", "flag_null_bene_id", "flag_negative_amount",
    "flag_zero_amount",   "flag_excessive_amount", "flag_future_date",
    "flag_null_service_date", "flag_null_icd10", "flag_bronze_null",
]

claims_flagged = (
    claims_deduped
    .withColumn("flag_null_claim_id",     F.col("claim_id").isNull())
    .withColumn("flag_null_bene_id",      F.col("bene_id").isNull())
    .withColumn("flag_negative_amount",   F.col("claim_amount") < 0)
    .withColumn("flag_zero_amount",       F.col("claim_amount") == 0)
    .withColumn("flag_excessive_amount",  F.col("claim_amount") > 500_000)
    .withColumn("flag_future_date",       F.col("service_date") > F.current_date())
    .withColumn("flag_null_service_date", F.col("service_date").isNull())
    .withColumn("flag_null_icd10",        F.col("icd10_primary").isNull())
    .withColumn("flag_bronze_null",       F.col("_bronze_null_flag") == True)
)

# Composite _quality_pass — one expression, single DAG step
combined_fail = F.lit(False)
for fc in FLAG_COLS:
    combined_fail = combined_fail | F.col(fc)

clean_claims = (
    claims_flagged
    .withColumn("_quality_pass", ~combined_fail)
    # Service feature engineering
    .withColumn("period",
        F.when(F.year("service_date") >= 2023, "post").otherwise("pre"))
    .withColumn("service_quarter", F.quarter("service_date"))
    .withColumn("is_inpatient",    (F.col("service_type") == "ip_admit").cast("int"))
    .withColumn("is_ed_visit",     (F.col("service_type") == "ed_visit").cast("int"))
    .withColumn("is_specialist",   (F.col("service_type") == "specialist").cast("int"))
    .withColumn("is_primary_care", (F.col("service_type") == "primary").cast("int"))
    .withColumn("cost_tier",
        F.when(F.col("claim_amount") < 500,   "low")
         .when(F.col("claim_amount") < 5000,  "medium")
         .otherwise("high"))
    .withColumn("_pipeline_layer", F.lit("silver"))
    .withColumn("_ingested_at",    F.current_timestamp())
)
print("✅ clean_claims transform defined (lazy DAG — not yet executed)")


In [0]:
# Write — single action triggers the full DAG
(
    clean_claims.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("claim_year", "claim_month")
    .saveAsTable(f"{CATALOG}.{SILVER_SCHEMA}.clean_claims")
)
spark.sql(
    f"OPTIMIZE {CATALOG}.{SILVER_SCHEMA}.clean_claims ZORDER BY (bene_id)"
)
print(f"✅ {CATALOG}.{SILVER_SCHEMA}.clean_claims written and optimized")


## 6. Quality validation

In [0]:
clean = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.clean_claims")
total      = clean.count()
pass_count = clean.filter("_quality_pass = true").count()
fail_count = total - pass_count

print(f"Total rows   : {total:,}")
print(f"Quality pass : {pass_count:,} ({pass_count/total*100:.1f}%)")
print(f"Quality fail : {fail_count:,} ({fail_count/total*100:.1f}%) — retained for audit")

# Per-flag breakdown
flag_counts = clean.agg(*[
    F.sum(F.col(c).cast("int")).alias(c) for c in FLAG_COLS
])
print("\n=== Flag breakdown ===")
display(flag_counts)


## 7. HCC v28 mapping — clinical enrichment

Applies the 5 `pandas_udf` closures to quality-pass claims.
Adds CMS v28 interaction terms: CHF×AFib (+0.175), CHF×Diabetes (+0.156), CKD×Diabetes (+0.156).


In [0]:
quality_claims = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.clean_claims").filter("_quality_pass = true")

hcc_mapped = (
    quality_claims
    # Core HCC enrichment via pandas_udf closures
    .withColumn("hcc_list",         icd_to_hcc_list("icd10_codes"))
    .withColumn("hcc_raf_total",     icd_to_raf_total("icd10_codes"))
    .withColumn("primary_hcc",       icd_to_primary_hcc("icd10_primary"))
    .withColumn("primary_hcc_desc",  icd_to_primary_hcc_desc("icd10_primary"))
    .withColumn("hcc_burden_count",  F.size("hcc_list"))
    # Boolean condition flags (array_contains — distributed, no UDF needed)
    .withColumn("has_chf",
        F.array_contains("hcc_list", 85))
    .withColumn("has_afib",
        F.array_contains("hcc_list", 96))
    .withColumn("has_diabetes",
        F.array_contains("hcc_list", 17) |
        F.array_contains("hcc_list", 18) |
        F.array_contains("hcc_list", 19))
    .withColumn("has_ckd",
        F.array_contains("hcc_list", 134) | F.array_contains("hcc_list", 135) |
        F.array_contains("hcc_list", 136) | F.array_contains("hcc_list", 137) |
        F.array_contains("hcc_list", 138))
    .withColumn("has_cancer",
        F.array_contains("hcc_list", 8)  | F.array_contains("hcc_list", 9)  |
        F.array_contains("hcc_list", 11) | F.array_contains("hcc_list", 12))
    .withColumn("has_copd",       F.array_contains("hcc_list", 111))
    .withColumn("has_depression",  F.array_contains("hcc_list", 58))
    .withColumn("has_metastatic",  F.array_contains("hcc_list", 8))
    # CMS v28 interaction terms
    .withColumn("interaction_chf_afib",
        (F.col("has_chf") & F.col("has_afib")).cast("double") * 0.175)
    .withColumn("interaction_chf_diabetes",
        (F.col("has_chf") & F.col("has_diabetes")).cast("double") * 0.156)
    .withColumn("interaction_ckd_diabetes",
        (F.col("has_ckd") & F.col("has_diabetes")).cast("double") * 0.156)
    .withColumn("hcc_interaction_total",
        F.col("interaction_chf_afib") +
        F.col("interaction_chf_diabetes") +
        F.col("interaction_ckd_diabetes"))
    .withColumn("_pipeline_layer", F.lit("silver"))
)

(
    hcc_mapped.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("claim_year", "claim_month")
    .saveAsTable(f"{CATALOG}.{SILVER_SCHEMA}.hcc_mapped_claims")
)
spark.sql(
    f"OPTIMIZE {CATALOG}.{SILVER_SCHEMA}.hcc_mapped_claims "
    f"ZORDER BY (bene_id, hcc_burden_count)"
)
print(f"✅ {CATALOG}.{SILVER_SCHEMA}.hcc_mapped_claims written and optimized")


## 8. Condition prevalence summary

In [0]:
hcc = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.hcc_mapped_claims")
print(f"HCC-mapped rows: {hcc.count():,}")
print()

display(
    hcc.agg(
        F.countDistinct("bene_id").alias("unique_members"),
        F.sum(F.col("has_chf").cast("int")).alias("n_chf"),
        F.sum(F.col("has_afib").cast("int")).alias("n_afib"),
        F.sum(F.col("has_diabetes").cast("int")).alias("n_diabetes"),
        F.sum(F.col("has_ckd").cast("int")).alias("n_ckd"),
        F.sum(F.col("has_cancer").cast("int")).alias("n_cancer"),
        F.sum(F.col("has_copd").cast("int")).alias("n_copd"),
        F.sum(F.col("has_metastatic").cast("int")).alias("n_metastatic"),
        F.round(F.avg("hcc_raf_total"), 3).alias("mean_hcc_raf"),
        F.round(F.avg("hcc_burden_count"), 2).alias("mean_hcc_burden"),
        F.round(F.avg("hcc_interaction_total"), 3).alias("mean_interaction_raf"),
    )
)


In [0]:
# Top conditions by member prevalence
print("=== Top 15 HCC conditions ===")
display(
    hcc
    .filter("primary_hcc_desc != 'Not Mapped' AND primary_hcc_desc IS NOT NULL")
    .groupBy("primary_hcc_desc", "primary_hcc")
    .agg(
        F.countDistinct("bene_id").alias("n_members"),
        F.count("claim_id").alias("n_claims"),
        F.round(F.avg("hcc_raf_total"), 3).alias("mean_hcc_raf"),
        F.round(F.sum("claim_amount"), 0).alias("total_cost_usd"),
    )
    .orderBy(F.desc("n_members"))
    .limit(15)
)


## 9. Member profile — demographics + demographic RAF

In [0]:
member_profile = (
    bronze_members
    .withColumn("enrollment_date", F.to_date("enrollment_date", "yyyy-MM-dd"))
    .withColumn("age",             F.col("age").cast("int"))
    .withColumn("dual_eligible",   F.col("dual_eligible").cast("int"))
    .withColumn("demographic_raf", demographic_raf_udf("sex", "age"))
    .withColumn("age_bracket",     (F.floor(F.col("age") / 5) * 5).cast("int"))
    .withColumn("flag_invalid_age",
        (F.col("age") < 0) | (F.col("age") > 120))
    .withColumn("_pipeline_layer", F.lit("silver"))
    .withColumn("_ingested_at",    F.current_timestamp())
)

(
    member_profile.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.{SILVER_SCHEMA}.member_profile")
)
print(f"✅ {CATALOG}.{SILVER_SCHEMA}.member_profile — {member_profile.count():,} members")
display(
    member_profile.groupBy("sex")
    .agg(
        F.count("*").alias("n"),
        F.round(F.avg("age"), 1).alias("mean_age"),
        F.round(F.avg("demographic_raf"), 3).alias("mean_demo_raf"),
        F.sum("dual_eligible").alias("n_dual_eligible"),
    )
)


## 10. Delta time travel demo

In [0]:
# Read version 0 of clean_claims (original write state)
v0 = (
    spark.read.format("delta")
    .option("versionAsOf", 0)
    .table(f"{CATALOG}.{SILVER_SCHEMA}.hcc_mapped_claims")
)
print(f"Version 0 row count: {v0.count():,}")
print()
print("=== Table history ===")
display(spark.sql(f"DESCRIBE HISTORY {CATALOG}.{SILVER_SCHEMA}.hcc_mapped_claims LIMIT 3"))


## ✅ Silver complete

| Table | Rows | Status |
|-------|------|--------|
| `silver.clean_claims` | ~1M | ✅ Deduplicated, 9-flag quality-validated |
| `silver.hcc_mapped_claims` | ~990K | ✅ HCC v28 enriched, RAF + interaction terms |
| `silver.member_profile` | 10,000 | ✅ Demographics + demographic RAF |

**Design note**: `spark.sparkContext.addPyFile()` is never called. HCC mapping runs via
`@pandas_udf` closures — the `_HCC` dict is captured on the driver and serialized
with each closure to workers. No JVM access required.

**Next**: Run `03_gold_aggregates` →
